# Stage 8A — Full Deterministic Python Environment Parity

Artık parçaları tek tek değil, **tek çağrıda baştan sona** test ediyoruz.

Python girdisi:

\[
\{\text{scenario}_{BR},\text{scenario}_{RU},f_c,
gNB,RIS,UE,\text{array sizes},WIdx,Z\}
\]

Python kendi başına:

\[
geometry
\rightarrow LSP/K
\rightarrow \mu_H,\sigma_H^2
\rightarrow \rho
\rightarrow W
\rightarrow \gamma
\rightarrow UBR
\rightarrow \mu_{Feff},\sigma^2_{Feff}
\rightarrow C
\rightarrow \mu_{SNR},\sigma^2_{Wick}.
\]

Bu aşamada stochastic pilot channel henüz yoktur. `W`, Type-I rank-1
`WIdx=[i11,i12,i2]` ile temsil edilir.

8 propagation condition test edilir ve her bankada 16 RIS candidate aynı GPU batch'inde değerlendirilir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, zipfile, shutil, json
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

required_modules = [
    'ris_gpu_geometry_lsp_stage67.py',
    'ris_gpu_physics_stage1.py',
    'ris_gpu_rho_stage2.py',
    'ris_gpu_ris_response_stage4.py',
    'ris_gpu_precoder_stage5.py',
    'ris_gpu_stats_stage3.py',
    'ris_gpu_environment_stage8.py',
]

# Add both likely locations to sys.path.
for d in [ROOT,Path('/content')]:
    if d.exists() and str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing = []
for name in required_modules:
    if not (ROOT/name).exists() and not (Path('/content')/name).exists():
        missing.append(name)

assert not missing, (
    "Şu Python modülleri bulunamadı:\n" + "\n".join(missing)
)

from ris_gpu_environment_stage8 import compare_stage8_case

print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :",torch.cuda.get_device_name(0))
print("Stage 8 module loaded.")

## MATLAB golden suite

MATLAB'da:

```matlab
export_stage8_full_deterministic_suite
```

çalıştır.

Default input:

```text
generate_train_scenarios_data_2000.csv
```

Çıktı:

```text
stage8_full_deterministic_golden.zip
```

Bunu Colab `/content` altına yükle.

In [ ]:
ZIP = Path('/content/stage8_full_deterministic_golden.zip')
assert ZIP.exists(), (
    "stage8_full_deterministic_golden.zip dosyasını /content altına yükle."
)

EXTRACT = Path('/content/stage8_full_deterministic_extract')
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(ZIP,'r') as zf:
    zf.extractall(EXTRACT)

mans = list(EXTRACT.rglob('manifest.csv'))
assert len(mans)==1,mans

SUITE = mans[0].parent
manifest = pd.read_csv(mans[0])

display(manifest)

assert len(manifest)==8
assert (manifest['nCandidates']==16).all()

print("PASS: Stage-8 suite structure")

In [ ]:
# DOUBLE / COMPLEX128 END-TO-END PARITY

device = 'cuda' if torch.cuda.is_available() else 'cpu'

rows = []

for _,meta in manifest.iterrows():
    print(
        f"Running {meta['caseName']} "
        f"(nT={meta['nT']}, nR={meta['nR']}, nRIS={meta['nRIS']}) ..."
    )

    m = compare_stage8_case(
        str(SUITE/str(meta['file'])),
        device=device,
        parity=True,
        gh_pair_chunk=80,
    )
    rows.append(m)

df64 = pd.DataFrame(rows)

metric_cols = [
    'scenarioBR','nT','nR','nRIS','nCandidates',
    'W_relFro',
    'gammaCandidates_relFro',
    'UBR_relFro',
    'muFeffCandidates_relFro',
    'sigma2FeffCandidates_relFro',
    'CmatCandidates_relFro',
    'muSNRCandidates_relFro',
    'sigma2WickCandidates_relFro',
]

display(df64[metric_cols])

rel_cols = [
    c for c in df64.columns
    if c.endswith('_relFro')
]

worst64 = df64[rel_cols].to_numpy().max()
where64 = np.unravel_index(
    np.argmax(df64[rel_cols].to_numpy()),
    df64[rel_cols].shape
)

print(
    "Worst double:",
    df64.iloc[where64[0]]['scenarioBR'],
    rel_cols[where64[1]],
    worst64
)

assert worst64 < 1e-10, (
    f"Stage 8 double end-to-end parity failed: {worst64:.3e}"
)

print("PASS: Stage 8A full deterministic double parity")

In [ ]:
# FLOAT32 / COMPLEX64 PRODUCTION END-TO-END SANITY

rows = []

for _,meta in manifest.iterrows():
    print(f"Running float32 {meta['caseName']} ...")

    m = compare_stage8_case(
        str(SUITE/str(meta['file'])),
        device=device,
        parity=False,
        gh_pair_chunk=80,
    )
    rows.append(m)

df32 = pd.DataFrame(rows)

display(df32[metric_cols])

rel_cols = [
    c for c in df32.columns
    if c.endswith('_relFro')
]

worst32 = df32[rel_cols].to_numpy().max()
where32 = np.unravel_index(
    np.argmax(df32[rel_cols].to_numpy()),
    df32[rel_cols].shape
)

print(
    "Worst float32:",
    df32.iloc[where32[0]]['scenarioBR'],
    rel_cols[where32[1]],
    worst32
)

assert worst32 < 1e-4, (
    f"Stage 8 float32 end-to-end sanity failed: {worst32:.3e}"
)

print("PASS: Stage 8A full deterministic float32 sanity")

In [ ]:
# Bottleneck profile — production float32
time_cols = [
    'scenarioBR','nRIS','nR',
    'time_geometry_lsp_s',
    'time_moments_s',
    'time_rho_s',
    'time_codebook_w_s',
    'time_prepare_w_state_s',
    'time_ris_response_s',
    'time_candidate_stats_s',
]

display(df32[time_cols])

summary = {
    c: float(df32[c].median())
    for c in time_cols
    if c.startswith('time_')
}

print("Median time per stage:")
print(json.dumps(summary,indent=2))

In [ ]:
# Save validation tables for later GitHub tests/README.
OUT64 = Path('/content/stage8_double_summary.csv')
OUT32 = Path('/content/stage8_float32_summary.csv')

df64.to_csv(OUT64,index=False)
df32.to_csv(OUT32,index=False)

print(OUT64)
print(OUT32)

## Stage 8A geçerse

Şu iddia artık test edilmiş olacaktır:

\[
\boxed{
\text{raw bank metadata}+WIdx+Z
\rightarrow
\mu_{SNR},\sigma^2_{Wick}
}
\]

MATLAB ara değişkeni olmadan Python tarafından baştan sona üretilmektedir.

Sonraki adım Stage 8B / stochastic path:

\[
h_{BR},h_{RU}\rightarrow F\rightarrow H_{\rm pilot}
\rightarrow SVD\rightarrow W
\]

ve ardından empirical `varEmp` üretiminin GPU'ya taşınmasıdır.